# 02 - Data Preprocessing & Leakage-Safe Pipeline Setup

## Objectives (Week 3 Deliverables)
1. **Chronological Splitting**: Partition data strictly by transaction timeline. Hold out the latest complete month as the test set and allocate preceding historical months as the training set (no random shuffling).
2. **Train-Only Outlier Filtering**: Compute the 0.5th and 99.5th percentile cutoffs for `ClosePrice` exclusively on the training split, freezing those boundaries to filter test data without distribution leakage.
3. **Feature Isolation**: Separate the target (`ClosePrice`), exclude structural metadata (`CloseDate`, `ListingKey`, invariant types), and isolate core physical drivers for the Week 4 baseline model (`LivingArea`, `BedroomsTotal`, `BathroomsTotalInteger`, `LotSizeSquareFeet`, `YearBuilt`).
4. **Leakage-Safe Preprocessing Pipeline**: Construct a `scikit-learn` `Pipeline` / `ColumnTransformer` encapsulating median imputation and standard scaling, fitted strictly on $X_{train}$ and applied to $X_{test}$.
5. **Artifact Export**: Export processed arrays and cleaned split datasets to disk for modeling.

In [1]:
import os
import glob
from pathlib import Path
import joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Styling & display options
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: "%.2f" % x)

print("Libraries imported successfully.")

Libraries imported successfully.


## Step 1: Load Scoped & Cleaned Dataset

We load the candidate dataset created during exploratory analysis, ensuring `CloseDate` is parsed as a datetime and the records strictly reflect single-family residential properties.

In [2]:
# Ingest candidate dataset or monthly extracts with candidate columns
CANDIDATE_COLUMNS = [
    "ListingKey", "CloseDate", "ClosePrice", "PropertyType", "PropertySubType",
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet",
    "YearBuilt", "Stories", "GarageSpaces", "Latitude", "Longitude",
    "PostalCode", "City", "PoolPrivateYN", "FireplaceYN"
]

data_dir = Path("../Sold files")
file_paths = sorted(glob.glob(str(data_dir / "CRMLSSold*.csv")))

dfs = []
for p in file_paths:
    header = pd.read_csv(p, nrows=1).columns
    cols = [c for c in CANDIDATE_COLUMNS if c in header]
    chunk = pd.read_csv(p, usecols=cols, low_memory=False)
    
    if "CloseDate" in chunk.columns:
        chunk["CloseDate"] = pd.to_datetime(chunk["CloseDate"], errors="coerce")
        
    mask = (
        (chunk["PropertyType"] == "Residential") &
        (chunk["PropertySubType"] == "SingleFamilyResidence") &
        (chunk["ClosePrice"] > 0) &
        (chunk["LivingArea"] > 0)
    )
    dfs.append(chunk[mask])

df = pd.concat(dfs, axis=0, ignore_index=True)
df = df.sort_values("CloseDate").reset_index(drop=True)

# Drop any duplicate transaction records by ListingKey
if "ListingKey" in df.columns:
    df = df.drop_duplicates(subset=["ListingKey"], keep="first").reset_index(drop=True)

print(f"Total sanitized Single-Family Residential listings loaded: {len(df):,}")
print(f"Dataset date range: {df['CloseDate'].min().strftime('%Y-%m-%d')} to {df['CloseDate'].max().strftime('%Y-%m-%d')}")

Total sanitized Single-Family Residential listings loaded: 309,177
Dataset date range: 2024-01-01 to 2026-04-30


## Step 2: Chronological Train / Test Split

A production AVM must be evaluated chronologically: train on historical data and predict future transactions.
* **Test Set**: The single most recent complete month available in the timeline.
* **Training Set**: The preceding historical months.

In [3]:
# Identify the latest complete month
latest_year = df["CloseDate"].dt.year.max()
latest_month = df[df["CloseDate"].dt.year == latest_year]["CloseDate"].dt.month.max()

test_start_date = pd.Timestamp(year=latest_year, month=latest_month, day=1)
print(f"Chronological split boundary: Test starts on {test_start_date.strftime('%Y-%m-%d')}")

# Create train and test partitions
train_mask = df["CloseDate"] < test_start_date
test_mask = df["CloseDate"] >= test_start_date

train_df = df[train_mask].copy().reset_index(drop=True)
test_df = df[test_mask].copy().reset_index(drop=True)

print(f"Training observations: {len(train_df):,} ({len(train_df) / len(df) * 100:.1f}%)")
print(f"Testing observations:  {len(test_df):,} ({len(test_df) / len(df) * 100:.1f}%)")

Chronological split boundary: Test starts on 2026-04-01
Training observations: 296,703 (96.0%)
Testing observations:  12,474 (4.0%)


## Step 3: Train-Only Target Outlier Thresholding

To avoid lookahead bias and distribution leakage:
1. Calculate the **0.5th** and **99.5th** percentiles of `ClosePrice` using `train_df` **only**.
2. Freeze these exact dollar cutoffs.
3. Apply the frozen cutoffs to filter both `train_df` and `test_df`. Do not recompute percentiles on the test set.

In [4]:
# Compute outlier thresholds on train_df ONLY
low_cutoff = float(train_df["ClosePrice"].quantile(0.005))
high_cutoff = float(train_df["ClosePrice"].quantile(0.995))

print(f"Frozen 0.5th Percentile Cutoff (Train): ${low_cutoff:,.2f}")
print(f"Frozen 99.5th Percentile Cutoff (Train): ${high_cutoff:,.2f}")

# Filter training set
n_train_before = len(train_df)
train_df = train_df[(train_df["ClosePrice"] >= low_cutoff) & (train_df["ClosePrice"] <= high_cutoff)].copy().reset_index(drop=True)
print(f"Train filtered: {len(train_df):,} rows retained ({n_train_before - len(train_df):,} outliers dropped)")

# Filter test set using the EXACT SAME training thresholds
n_test_before = len(test_df)
test_df = test_df[(test_df["ClosePrice"] >= low_cutoff) & (test_df["ClosePrice"] <= high_cutoff)].copy().reset_index(drop=True)
print(f"Test filtered:  {len(test_df):,} rows retained ({n_test_before - len(test_df):,} outliers dropped)")

Frozen 0.5th Percentile Cutoff (Train): $190,000.00
Frozen 99.5th Percentile Cutoff (Train): $8,244,900.00
Train filtered: 293,758 rows retained (2,945 outliers dropped)
Test filtered:  12,336 rows retained (138 outliers dropped)


## Step 4: Feature Matrix & Target Vector Isolation

We isolate the ground-truth target vector `ClosePrice` into `y_train` and `y_test`. 

For the initial **Week 4 Baseline Linear Regression**, we isolate the core continuous physical drivers:
* `LivingArea`
* `BedroomsTotal`
* `BathroomsTotalInteger`
* `LotSizeSquareFeet`
* `YearBuilt`

Remaining geographic and amenity attributes are preserved in the tabular datasets for downstream feature engineering (Weeks 5–7).

In [5]:
# Define baseline feature subset
BASELINE_NUMERIC_FEATURES = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt"
]

TARGET = "ClosePrice"

# Separate X and y
X_train_raw = train_df[BASELINE_NUMERIC_FEATURES].copy()
y_train = train_df[TARGET].values

X_test_raw = test_df[BASELINE_NUMERIC_FEATURES].copy()
y_test = test_df[TARGET].values

print(f"X_train_raw shape: {X_train_raw.shape}, y_train shape: {y_train.shape}")
print(f"X_test_raw shape:  {X_test_raw.shape}, y_test shape:  {y_test.shape}")

X_train_raw shape: (293758, 5), y_train shape: (293758,)
X_test_raw shape:  (12336, 5), y_test shape:  (12336,)


## Step 5: Construct Leakage-Safe Preprocessing Pipeline

We construct an encapsulated `Pipeline` using `SimpleImputer` (median strategy) and `StandardScaler`. 

* The pipeline is fitted **only on `X_train_raw`**.
* The fitted pipeline transforms `X_test_raw` without ever exposing test statistics to the imputer medians or scaling means/variances.

In [6]:
# Define preprocessing pipeline for numeric predictors
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Fit on training data ONLY and transform both partitions
X_train_processed = numeric_pipeline.fit_transform(X_train_raw)
X_test_processed = numeric_pipeline.transform(X_test_raw)

# Verification assertions
assert not np.isnan(X_train_processed).any(), "NaN values found in X_train_processed!"
assert not np.isnan(X_test_processed).any(), "NaN values found in X_test_processed!"
assert X_train_processed.shape[1] == len(BASELINE_NUMERIC_FEATURES), "Feature count mismatch in train!"
assert X_test_processed.shape[1] == len(BASELINE_NUMERIC_FEATURES), "Feature count mismatch in test!"

print("Pipeline fit and transform completed successfully with zero nulls.")
print(f"X_train_processed: {X_train_processed.shape}")
print(f"X_test_processed:  {X_test_processed.shape}")

Pipeline fit and transform completed successfully with zero nulls.
X_train_processed: (293758, 5)
X_test_processed:  (12336, 5)


## Step 6: Verify Distribution Normalization & Consistency

Checking means and standard deviations of processed features to ensure standard scaling centered training inputs around $\mu \approx 0$ and $\sigma \approx 1$.

In [7]:
train_processed_df = pd.DataFrame(X_train_processed, columns=BASELINE_NUMERIC_FEATURES)
test_processed_df = pd.DataFrame(X_test_processed, columns=BASELINE_NUMERIC_FEATURES)

verification_summary = pd.DataFrame({
    "Train Mean": train_processed_df.mean(),
    "Train Std": train_processed_df.std(),
    "Test Mean": test_processed_df.mean(),
    "Test Std": test_processed_df.std()
})

display(verification_summary)

,Train Mean,Train Std,Test Mean,Test Std
LivingArea,0.00,1.00,0.02,0.99
BedroomsTotal,0.00,1.00,0.00,0.99
BathroomsTotalInteger,0.00,1.00,0.03,0.94
LotSizeSquareFeet,-0.00,1.00,-0.01,0.34
YearBuilt,-0.00,1.00,0.02,1.00


## Step 7: Export Cleaned Datasets & Transformation Artifacts

Save the processed arrays, unscaled target vectors, full cleaned DataFrames, and fitted pipeline object to disk for immediate loading in baseline_model.

In [8]:
# Create artifacts directory
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

# 1. Save fitted pipeline
joblib.dump(numeric_pipeline, artifacts_dir / "baseline_preprocessing_pipeline.joblib")

# 2. Save numpy arrays for fast baseline model training
np.save(artifacts_dir / "X_train_processed.npy", X_train_processed)
np.save(artifacts_dir / "X_test_processed.npy", X_test_processed)
np.save(artifacts_dir / "y_train.npy", y_train)
np.save(artifacts_dir / "y_test.npy", y_test)

# 3. Save full cleaned DataFrames with preserved spatial/amenity columns for Weeks 5-7
train_df.to_parquet(artifacts_dir / "train_cleaned.parquet", index=False)
test_df.to_parquet(artifacts_dir / "test_cleaned.parquet", index=False)

print(f"All Week 3 preprocessing artifacts exported to '{artifacts_dir.resolve()}':")
for file in sorted(artifacts_dir.glob("*")):
    print(f" - {file.name} ({file.stat().st_size / (1024**2):.2f} MB)")

All Week 3 preprocessing artifacts exported to '/Users/danielmuhire/Desktop/IDX Exchange Files/Deliverables/artifacts':
 - X_test_processed.npy (0.47 MB)
 - X_train_processed.npy (11.21 MB)
 - baseline_preprocessing_pipeline.joblib (0.00 MB)
 - test_cleaned.parquet (0.50 MB)
 - train_cleaned.parquet (10.09 MB)
 - y_test.npy (0.09 MB)
 - y_train.npy (2.24 MB)


## Step 8: Execution Verification, Output Summary & Handoff to Week 4

### 1. Preprocessing Summary & Output Verification
All data preparation steps for Week 3 have executed successfully, enforcing strict leakage boundaries across the single-family residential dataset:

* **Chronological Partitioning**:
  * **Training Split**: Encompasses historical transactions prior to the cutoff boundary, capturing seasonality and core market trends across preceding months.
  * **Testing Split**: The single most recent complete month was held out as the evaluation set without random shuffling or future-data leakage.
* **Train-Only Outlier Filtering**:
  * The 0.5th and 99.5th percentile thresholds of `ClosePrice` were computed exclusively on `train_df`.
  * The frozen boundaries were applied uniformly to both partitions, removing extreme luxury artifacts and price-entry errors without exposing test distribution statistics.
* **Encapsulated Preprocessing Pipeline**:
  * Built a `scikit-learn` `Pipeline` utilizing `SimpleImputer(strategy='median')` and `StandardScaler()`.
  * The pipeline was fitted strictly on `X_train_raw` and transformed onto `X_test_raw`, ensuring zero lookahead bias.
  * Confirmed zero missing (`NaN`) values remain across processed feature matrices.

---

### 2. Exported Artifacts Ledger
The following production artifacts have been saved to the `artifacts/` directory and are locked for downstream modeling:

| Artifact | Type | Dimensions / Contents | Downstream Milestone |
| :--- | :--- | :--- | :--- |
| `baseline_preprocessing_pipeline.joblib` | Scikit-Learn Pipeline | Median Imputer + Standard Scaler | Inference & App Deployment (Week 9) |
| `X_train_processed.npy` | NumPy Matrix | Processed 5-feature baseline predictors (Train) | Baseline Linear Regression (Week 4) |
| `X_test_processed.npy` | NumPy Matrix | Processed 5-feature baseline predictors (Test) | Baseline Evaluation & Test R² (Week 4) |
| `y_train.npy` | NumPy Array | Raw `ClosePrice` target values (Train) | Model Training (Week 4) |
| `y_test.npy` | NumPy Array | Raw `ClosePrice` target values (Test) | Ground-Truth Metric Scrutiny (Week 4) |
| `train_cleaned.parquet` | Parquet Table | Complete sanitized training rows (18 candidate columns) | Feature Engineering & GeoPandas (Weeks 5–7) |
| `test_cleaned.parquet` | Parquet Table | Complete sanitized testing rows (18 candidate columns) | Feature Engineering & GeoPandas (Weeks 5–7) |

---

### 3. Concrete Next Steps for Week 4 (Baseline Modeling)
1. Initialize `03_baseline_model.ipynb`.
2. Load the pre-scaled training and testing arrays (`X_train_processed.npy`, `y_train.npy`, `X_test_processed.npy`, `y_test.npy`) directly from `artifacts/`.
3. Train an Ordinary Least Squares `LinearRegression()` model to establish the benchmark fit on core physical features (`LivingArea`, `BedroomsTotal`, `BathroomsTotalInteger`, `LotSizeSquareFeet`, `YearBuilt`).
4. Evaluate baseline metrics ($R^2$, MAE, and MdAPE) on the held-out test split to set the minimum performance threshold that advanced tree and gradient-boosted ensembles must beat in subsequent weeks.